# 2 · From score to policy 🩺

Yesterday you built a model that gives every appointment a **no-show score**. Today you turn that score into **decisions** Lena can use on Monday morning.

**What you practise:**
- Working with **scores** (probabilities) instead of yes/no labels, and checking they're honest (Day 1).
- Asking: **do we even need a model** for this?
- **Capacity**: who to call when you can only make 300 calls (top-k).
- **Money**: the threshold where an action pays off, and how it moves when the costs change.
- **Fairness**: who carries the cost of the model's mistakes?
- A **ship / pilot / improve** recommendation with a KPI and a guardrail.

> **How this notebook works**
> - Run the cells **from top to bottom** with `Shift + Enter`.
> - ▶️ **Run** cells work as they are.
> - ✏️ **Your turn** cells contain `___` (three underscores). Replace each `___` with your answer, then run the cell.
> - ✅ **Check** cells tell you whether you got it right. If a check fails, read the hint, fix the cell above and run both again.
> - 🧠 **Think** cells ask a PM question. Double-click the answer cell below it and write 1–3 sentences.
> - If you see `NameError: name '___' is not defined`, a blank is still empty.
> - Stuck? The `solutions/` folder has the same notebook, solved.

## Step 0 · Get back to where we stopped

In [ ]:
# ▶️ Run: yesterday's notebook in one cell (load → date features → split by date → recipe → logistic regression)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

appointments = pd.read_csv("data/praxisnow_appointments.csv", parse_dates=["booked_at", "appointment_at"])
appointments["lead_days"] = (appointments["appointment_at"].dt.normalize() - appointments["booked_at"]).dt.days
appointments["weekday"] = appointments["appointment_at"].dt.day_name()
appointments["hour"] = appointments["appointment_at"].dt.hour

numeric = ["lead_days", "hour", "age", "distance_km", "sms_opt_in", "prior_appointments", "prior_no_shows"]
categorical = ["weekday", "clinic_city", "specialty", "booking_channel", "insurance"]
is_train = appointments["appointment_at"] < "2026-08-01"
train, test = appointments[is_train].copy(), appointments[~is_train].copy()


def make_model(num_cols, cat_cols):
    prep = ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median", add_indicator=True)),
                          ("scale", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
    return Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000))])


model = make_model(numeric, categorical).fit(train[numeric + categorical], train["no_show"])
print("Test appointments (Aug–Oct):", len(test), "| real no-shows:", test["no_show"].sum())
print("Test AUC:", round(roc_auc_score(test["no_show"], model.predict_proba(test[numeric + categorical])[:, 1]), 3))

**The three decisions again**, now with Lena's numbers. The test period (August–October) has 1,874 appointments.

| | What it means | Cost | Value | Limit |
|---|---|---|---|---|
| 📱 **SMS** | already sent 24 h before to everyone who **opted in**. Finance asks: *"text only the risky ones, to save money?"* | €0.10 each | reminds people | only patients who opted in |
| ☎️ **Call** | front desk phones the patient a few days before: *"Do you still need this appointment?"* | staff time | some patients come, some free the slot | **300 calls** in the test period (~25 a week) |
| 📅 **Double-book** | give the slot to a second patient from the waiting list as well | – | **+€60** if the first patient doesn't come (the slot is used) | **−€40** if both come (overtime, someone waits ~45 min) |

## Step 1 · Scores, not labels

In [ ]:
# ✏️ Your turn: get the no-show SCORE (probability of class 1) for every test appointment
# Hint: predict_proba returns two columns: P(came) and P(no-show).
test["score"] = model.predict_proba(test[numeric + categorical])[:, ___]
test[["appointment_at", "specialty", "lead_days", "prior_no_shows", "score", "no_show"]].sort_values("score", ascending=False).head()

In [ ]:
# ✅ Check
assert abs(test["score"].mean() - 0.2026) < 0.001, "Take column 1: the probability of a no-show"
print("✅ Correct: the average score is 0.20, close to the real no-show share (0.20).")

**Can we read a score of 0.3 as "30 % chance"?** The average matching is a good sign, but not proof. Let's check band by band: *when the model says about 30 %, do about 30 % really not come?*

In [ ]:
# ▶️ Run: are the scores honest? Average score vs real no-show share, per score band
band = pd.cut(test["score"], [0, 0.2, 0.4, 0.6, 1.0])
honesty = test.groupby(band, observed=True).agg(appointments=("score", "size"),
                                                model_says=("score", "mean"),
                                                really_no_show=("no_show", "mean")).round(2)
assert (honesty["model_says"] - honesty["really_no_show"]).abs().max() < 0.06   # (course check: bands stay close)
honesty

In these four bands the two numbers are close, so we may use the scores as **rough** probabilities. That matters in Step 4, where we multiply a score by euros.

> ⚠️ One detail the four bands hide: with smaller bands, the high scores are a bit **too confident** (when the model says 0.65, really 0.59 didn't come; at 0.76, really 0.65). Keep that in mind for high thresholds in Step 4.

In [ ]:
# ▶️ Run: the same check with ten smaller bands (bands with few appointments are noisier)
fine = test.groupby(pd.cut(test["score"], np.arange(0, 1.01, 0.1)), observed=True).agg(
    appointments=("score", "size"), model_says=("score", "mean"), really_no_show=("no_show", "mean")).round(2)
fine

In [ ]:
# ▶️ Run: how the scores spread, for patients who came vs no-shows
plt.figure(figsize=(9, 3.5))
plt.hist(test.loc[test["no_show"] == 0, "score"], bins=30, alpha=0.6, label="came", color="#3b7dd8")
plt.hist(test.loc[test["no_show"] == 1, "score"], bins=30, alpha=0.6, label="no-show", color="#d8553b")
plt.axvline(0.5, color="black", ls="--", label="default threshold 0.5")
plt.xlabel("no-show score"); plt.ylabel("appointments"); plt.legend()
plt.show()

The two groups overlap: **no threshold separates them perfectly**. Every threshold is a trade: more no-shows caught means more patients wrongly flagged. *Which* trade is right depends on the action.

## Step 2 · Decision 1: the SMS (Finance's question)

Today every patient who **opted in** gets the reminder SMS. Finance asks: *"Could we use the model to text only the risky patients and save money?"* First: how much money is there to save?

In [ ]:
# ✏️ Your turn: how many test appointments belong to patients who opted in, and what do their SMS cost?
opted_in = test["sms_opt_in"].___()
sms_cost = opted_in * ___
print(f"{opted_in} of {len(test)} appointments get an SMS today. Cost for three months: €{sms_cost:,.2f}")
print(f"For comparison: ONE empty chair costs €60. All {test['no_show'].sum()} no-shows cost €{test['no_show'].sum() * 60:,}.")

In [ ]:
# ✅ Check
assert opted_in == 1216 and abs(sms_cost - 121.6) < 0.01, "Sum the sms_opt_in column (it's 0/1), then multiply by €0.10"
print("✅ Correct: the whole SMS bill for three months is about the price of two empty chairs.")

🧠 **Think:** Should PraxisNow use the model to text only the risky patients? And may we text patients who did **not** opt in?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 3 · Decision 2: who gets one of the 300 calls?

In [ ]:
# ✏️ Your turn: call the 300 appointments with the HIGHEST scores
# A call happens a few days before, so only appointments booked at least 3 days ahead can get one.
callable_appts = test[test["lead_days"] >= 3]
calls = 300
call_list = callable_appts.sort_values("score", ascending=___).head(___)
reached = call_list["no_show"].___()
random_reached = calls * callable_appts["no_show"].mean()
print(f"{len(callable_appts)} of {len(test)} appointments can be called.")
print(f"The model's call list contains {reached} no-shows; 300 random calls (among the same callable appointments) would reach about {random_reached:.0f}.")

In [ ]:
# ✅ Check
assert reached == 160, "Sort by score, highest first (ascending=False), take 300, sum no_show"
print(f"✅ {reached} vs {random_reached:.0f}: the model's list reaches {reached / random_reached:.1f}× as many no-shows as random calls.")
print(f"   {reached / calls:.0%} of the calls go to a real no-show (precision), and the list holds {reached / test['no_show'].sum():.0%} of ALL no-shows in the test period.")

> ⚠️ **Reach is not prevention.** 160 is how many no-shows are **on the list**, not how many the calls will **stop**. Some patients will skip anyway after the call, and some will cancel too late for anyone else to take the slot. How many no-shows a call really prevents, only a **pilot** can tell (call half of the list, compare with the other half).

In [ ]:
# ▶️ Run: what if the front desk could make more (or fewer) calls?
ranked = callable_appts.sort_values("score", ascending=False)["no_show"].to_numpy()
capacity = pd.DataFrame({"calls": [100, 200, 300, 400, 600, 900]})
capacity["no-shows on the list"] = [int(ranked[:k].sum()) for k in capacity["calls"]]
capacity["random calls would reach"] = (capacity["calls"] * callable_appts["no_show"].mean()).round(0).astype(int)
capacity["hit rate"] = (capacity["no-shows on the list"] / capacity["calls"]).round(2)
capacity

🧠 **Think:** The front desk asks for a second person so they can make **600** calls instead of 300. Will that reach twice as many no-shows? What would you tell Lena?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 4 · Decision 3: double-booking, and where it pays off

Double-booking a slot is worth it when the expected gain beats the expected loss:

$$\text{score} \times 60 \;>\; (1 - \text{score}) \times 40 \quad\Longleftrightarrow\quad \text{score} > \frac{40}{60 + 40}$$

> 📌 **This rule only works if:** the scores are roughly honest probabilities (Step 1: roughly, yes), the €60 and €40 are good guesses, there really is a patient on the waiting list for that slot, and the doctors agree it's medically fine for that kind of appointment. Each of these is a reason to **pilot** it first.

In [ ]:
# ✏️ Your turn: compute the break-even threshold, then double-book every slot above it
slot_value, wrong_cost = 60, 40
break_even = wrong_cost / (___ + ___)
double_book = test["score"] >= ___

right = (double_book & (test["no_show"] == 1)).sum()   # slot used, +€60
wrong = (double_book & (test["no_show"] == 0)).sum()   # both came, −€40
print(f"Threshold {break_even:.2f}: double-book {double_book.sum()} slots → {right} right, {wrong} wrong")
print(f"Net value: €{right * slot_value - wrong * wrong_cost:,}")

In [ ]:
# ✅ Check
assert abs(break_even - 0.4) < 1e-9 and double_book.sum() == 280
print(f"✅ Correct. Out of €{test['no_show'].sum() * 60:,} lost to empty slots, double-booking wins back about €4,300 net (if our cost guesses are right).")
print(f"   Price of it: {wrong} patients who came had their slot double-booked.")

In [ ]:
# ▶️ Run: net value for many thresholds
def net_value(threshold, gain=60, cost=40):
    flag = test["score"] >= threshold
    return int(gain * (flag & (test["no_show"] == 1)).sum() - cost * (flag & (test["no_show"] == 0)).sum())

thresholds = np.round(np.arange(0.20, 0.76, 0.05), 2)
sweep = pd.DataFrame({"threshold": thresholds,
                      "double-booked": [(test["score"] >= t).sum() for t in thresholds],
                      "net € (wrong = €40)": [net_value(t) for t in thresholds]})
print(sweep.to_string(index=False))
assert sweep.loc[sweep["net € (wrong = €40)"].idxmax(), "threshold"] in (0.35, 0.40)   # (course check)

The best values in the table sit around **0.35–0.40**, right where the formula said. (Not exactly: the scores aren't perfect probabilities, and 1,874 appointments is a small sample, so the table is a bit noisy.)

> 📌 **Honesty note:** we are choosing a threshold by looking at the test months' outcomes, so these euros are a bit optimistic. In real life you'd take the threshold from the formula (or from an earlier period) and then confirm it on new data in a pilot.

Now **Dr. Okafor** objects: *"A patient waiting 45 minutes isn't a €40 problem. It's a complaint, maybe a bad review, maybe a patient we lose. Count it as **€80**."*

In [ ]:
# ✏️ Your turn: redo the calculation with Dr. Okafor's cost
okafor_cost = ___
okafor_break_even = okafor_cost / (slot_value + okafor_cost)
sweep["net € (wrong = €80)"] = [net_value(t, cost=okafor_cost) for t in thresholds]
print(f"New break-even threshold: {okafor_break_even:.2f}")
print(sweep.to_string(index=False))

In [ ]:
# ✅ Check
assert abs(okafor_break_even - 0.571) < 0.001
assert net_value(0.40, cost=80) < 0 < net_value(0.57, cost=80)
print(f"✅ Correct. With €80, the old threshold 0.40 gives a net of −€{abs(net_value(0.40, cost=80)):,}: it LOSES money.")
print(f"   At the new break-even 0.57 it's positive again (€{net_value(0.57, cost=80):,}), with far fewer double-bookings ({(test['score'] >= 0.57).sum()} instead of {(test['score'] >= 0.40).sum()}).")
print("   The table jumps around a bit between 0.55 and 0.75: small sample, and high scores are a bit too confident (Step 1).")
print("🏁 M4 reached: a call list for a fixed capacity and a money-based double-booking threshold.")

🧠 **Think:** The model didn't change, but the break-even threshold moved from 0.40 to 0.57. Who should decide whether a 45-minute wait "costs" €40 or €80: the data team, Lena, Dr. Okafor? What does this tell you about thresholds?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 5 · Who carries the cost of the mistakes?

A wrong double-booking doesn't hit "the practice" in general: it hits **a specific patient who came on time and then waited**. Let's check who those patients are, using the €40 threshold (0.40).

In [ ]:
# ▶️ Run: a helper that slices the decisions by group
test["double_booked"] = test["score"] >= 0.40
test["age_group"] = pd.cut(test["age"], [17, 29, 49, 64, 99], labels=["18-29", "30-49", "50-64", "65+"])


def by_group(group_col, flag_col="double_booked"):
    rows = {}
    for name, g in test.groupby(group_col, observed=True):
        came = g[g["no_show"] == 0]
        rows[name] = {"appointments": len(g),
                      "real no-show rate": g["no_show"].mean(),
                      "double-booked": g[flag_col].mean(),
                      "came but double-booked": came[flag_col].mean()}
    table = pd.DataFrame(rows).T.round(3)
    table["appointments"] = table["appointments"].astype(int)
    return table

In [ ]:
# ✏️ Your turn: slice by age group, then by insurance
print(by_group(___))
print()
print(by_group(___))

In [ ]:
# ✅ Check
young = by_group("age_group").loc["18-29", "came but double-booked"]
old = by_group("age_group").loc["65+", "came but double-booked"]
public = by_group("insurance").loc["public", "came but double-booked"]
private = by_group("insurance").loc["private", "came but double-booked"]
assert abs(young - 0.184) < 0.001 and abs(old - 0.022) < 0.001
assert public > private
print(f"✅ Of the young patients who CAME, {young:.1%} had their slot double-booked; of the 65+ patients, {old:.1%}: about {young / old:.0f}× as often.")
print(f"   Publicly insured patients who came: {public:.1%}; privately insured: {private:.1%}.")

🧠 **Think:** Young patients really do miss more appointments (28% vs 14% for 65+). So is it fair that the young patients who **do** come on time are so much more likely to wait? What would you propose?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

**Predict, then run:** if we simply **remove `age` and `insurance`** from the model, will the gap disappear? A) yes B) it shrinks C) no change

In [ ]:
# ▶️ Run: the same model without age and insurance
no_age_num = [c for c in numeric if c != "age"]
no_age_cat = [c for c in categorical if c != "insurance"]
model_no_age = make_model(no_age_num, no_age_cat).fit(train[no_age_num + no_age_cat], train["no_show"])
test["score_no_age"] = model_no_age.predict_proba(test[no_age_num + no_age_cat])[:, 1]
test["double_booked_no_age"] = test["score_no_age"] >= 0.40
print("Test AUC without age and insurance:", round(roc_auc_score(test["no_show"], test["score_no_age"]), 3), "(with them: 0.782)")
no_age = by_group("age_group", "double_booked_no_age")
print(no_age)
assert 1 < no_age.loc["18-29", "came but double-booked"] / no_age.loc["65+", "came but double-booked"] < young / old   # (course check: shrinks, doesn't vanish)

Answer: **B**. For patients who came, the gap shrinks a lot (18–29: 18.4% → 10.2%; 65+: 2.2% → 6.1%) but doesn't vanish. Notice the price: **older patients now wait more often** (65+: 2.2% → 6.1%), and the model gets a little worse (AUC 0.782 → 0.776).

In [ ]:
# ▶️ Run: why doesn't it vanish? Look at other columns by age group
test.groupby("age_group", observed=True).agg(
    books_in_app=("booking_channel", lambda s: (s == "app").mean()),
    prior_no_shows=("prior_no_shows", "mean"),
    sms_opt_in=("sms_opt_in", "mean")).round(2)

🧠 **Think:** Why didn't removing `age` remove the age gap completely? What does that mean for a PM who hears: *"The model is fair, we don't even use age!"*

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

## Step 6 · Your recommendation for Lena

Write it the way you would post it in Slack before the meeting. You present it in **3 minutes at the stand-up** (one person per group).

✏️ **Your turn:** double-click and fill in every line.

- **Recommendation** (ship / pilot / improve):
- **SMS:**
- **Calls (rule):**
- **Double-booking (rule):**
- **Expected effect** (one number from this notebook, and what it does NOT prove):
- **Main risk, and who carries it:**
- **One KPI** to prove it works:
- **One guardrail** that stops or pauses it:
- **What's still missing before a full rollout:**

In [ ]:
# ▶️ Run after writing your brief
print("🏁 M5 reached if your brief has: a rule per decision, one number (and what it doesn't prove), the main risk, a KPI and a guardrail.")
print("Bring it to the stand-up: 3 minutes per group.")

## Step 7 · Bonus: a second model, the day before

Remember `reminder_reply`, the leak from yesterday? The SMS goes out **24 hours before** the appointment. Imagine a second decision taken **12 hours before** the appointment, when the reply window has closed: "double-book tomorrow's risky slots". For **that** decision, the reply is known, so it's a fair feature.

*(Simplification: we assume every reply arrives within those first 12 hours.)*

In [ ]:
# ✏️ Your turn: add reminder_reply as a category ("no_sms" when no SMS was sent) and retrain
for df in (train, test):
    df["reply"] = df["reminder_reply"].fillna(___)
day_before_cat = categorical + [___]
model_day_before = make_model(numeric, day_before_cat).fit(train[numeric + day_before_cat], train["no_show"])
test["score_day_before"] = model_day_before.predict_proba(test[numeric + day_before_cat])[:, 1]
print("Test AUC at booking time:  ", round(roc_auc_score(test["no_show"], test["score"]), 3))
print("Test AUC 12 h before:      ", round(roc_auc_score(test["no_show"], test["score_day_before"]), 3))
flag = test["score_day_before"] >= 0.40
print("Double-booking net value (0.40, €40):",
      f"€{int(60 * (flag & (test['no_show'] == 1)).sum() - 40 * (flag & (test['no_show'] == 0)).sum()):,}", "(at booking time: €4,300)")

In [ ]:
# ✅ Check
assert abs(roc_auc_score(test["no_show"], test["score_day_before"]) - 0.848) < 0.002
print("✅ Correct: knowing who confirmed lifts the AUC from 0.78 to 0.85.")

🧠 **Think:** The same column was a leak yesterday and a good feature today. What changed? And what does this suggest for the **product**?

**✍️ Your answer:** *double-click here and write 1–3 sentences.*

### 🎉 Done!
You went from a raw table to **decisions**: a model you can explain, a rule per action (including "no model needed" for the SMS), a number for the business case and what it doesn't prove, a fairness check, and a recommendation with a KPI and a guardrail. That's the PM's half of a machine-learning product.